In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os

# Define the directory where your CSVs are saved
data_dir = '/path/to/xGATE/data/senescence_4/'

# Map the display titles to their corresponding file names (Titles include spaces)
file_mapping = [
    {'title': 'CTRL',   'filename': 'competitive_analysis_summary_ctrl.csv'},
    {'title': 'ETO 0',  'filename': 'competitive_analysis_summary_eto_0.csv'},
    {'title': 'ETO 1',  'filename': 'competitive_analysis_summary_eto_1.csv'},
    {'title': 'ETO 2',  'filename': 'competitive_analysis_summary_eto_2.csv'},
    {'title': 'ETO 4',  'filename': 'competitive_analysis_summary_eto_4.csv'},
    {'title': 'ETO 7',  'filename': 'competitive_analysis_summary_eto_7.csv'},
    {'title': 'ETO 10', 'filename': 'competitive_analysis_summary_eto_10.csv'}
]

def determine_activity_color(emp_cc, emp_sen, comp_cc, comp_sen):
    """
    Assign color based on statistical significance (empirical and competitive p-values).
    green = Cell Cycle, blue = Senescence, grey = Neither/Not Significant
    """
    # Fallback for empty sparse clusters
    if pd.isna(emp_cc) or pd.isna(emp_sen):
        return 'grey'
        
    # Case 1: Neither is significant
    if emp_cc >= 0.05 and emp_sen >= 0.05:
        return 'grey'
        
    # Case 2: Only Cell Cycle is significant
    elif emp_cc < 0.05 and emp_sen >= 0.05:
        return 'lightgreen'
        
    # Case 3: Only Senescence is significant
    elif emp_sen < 0.05 and emp_cc >= 0.05:
        return 'skyblue'
        
    # Case 4: Both are significant (Empirical p-vals < 0.05)
    elif emp_cc < 0.05 and emp_sen < 0.05:
        # Check competitive p-values to break the tie
        if comp_cc < 0.05:
            return 'lightgreen'
        elif comp_sen < 0.05:
            return 'skyblue'
        else:
            return 'grey' # Failsafe if neither competitive p-val is < 0.05
            
    return 'grey' # Ultimate fallback

# =========================
# Standalone Legend Generation
# =========================
print("Generating standalone legend...")
legend_patches = [
    mpatches.Patch(color='skyblue', label='Cellular Senescence'),
    mpatches.Patch(color='lightgreen', label='Cell Cycle'),
    mpatches.Patch(color='grey', label='Not Significant')  # Added grey patch
]

fig_leg = plt.figure(figsize=(10, 2), facecolor='white') # Made slightly wider to fit 3 items
ax_leg = fig_leg.add_subplot(111)
ax_leg.axis('off') # Hide axes for the legend image
ax_leg.legend(handles=legend_patches, loc='center', ncol=3, frameon=False, fontsize=18)

legend_save_path = os.path.join(data_dir, "Standalone_Legend.png")
fig_leg.savefig(legend_save_path, dpi=300, bbox_inches='tight', facecolor='white')
plt.close(fig_leg)
print(f"Legend saved to {legend_save_path}")


# =========================
# Individual Plot Generation
# =========================
print("\nLoading data and generating individual plots...")

for group in file_mapping:
    filepath = os.path.join(data_dir, group['filename'])
    
    if not os.path.exists(filepath):
        print(f"Warning: File {filepath} not found. Skipping {group['title']}.")
        continue
        
    print(f"Processing and plotting {group['title']}...")
    df = pd.read_csv(filepath)
    
    # Find the largest cluster in this dataset to act as our baseline for maximum bubble size
    max_cells = df['Number of cells'].max()
    
    # Set the area of the absolute largest bubble
    max_bubble_area = 2500 
    
    clusters_list = df['Cluster'].tolist()
    z_diff_list = []
    colors = []
    scatter_sizes = []
    
    for index, row in df.iterrows():
        z_diff = row['Z-Score Difference (Senescence - Cell Cycle)']
        num_cells = row['Number of cells']
        
        # Extract the p-values for the color logic
        emp_cc = row['Cell Cycle Empirical P-value']
        emp_sen = row['Senescence Empirical P-value']
        comp_cc = row['Competitive P-value Cell Cycle']
        comp_sen = row['Competitive P-value Senescence']
        
        # Handle potential NaNs from sparse clusters
        if pd.isna(z_diff):
            z_diff = 0
            
        # Cap the z_diff to ±10
        z_diff_capped = 10 if z_diff >= 10 else (-10 if z_diff <= -10 else z_diff)
        
        z_diff_list.append(z_diff_capped)
        
        # Apply the new p-value based color logic
        bubble_color = determine_activity_color(emp_cc, emp_sen, comp_cc, comp_sen)
        colors.append(bubble_color)
        
        # STRICT PROPORTIONALITY: 
        # Map the cell count linearly to the area ('s') based on the largest cluster.
        if max_cells > 0:
            proportional_area = (num_cells / max_cells) * max_bubble_area
        else:
            proportional_area = 0
            
        scatter_sizes.append(proportional_area)
        
    # Create the figure with your updated dimensions
    fig, ax = plt.subplots(figsize=(8, 6), facecolor='white')
    
    ax.scatter(
        z_diff_list, 
        clusters_list, 
        s=scatter_sizes, 
        c=colors, 
        alpha=0.6, 
        edgecolors='black'
    )
    
    # Set the title dynamically with the space included
    ax.set_title(f"{group['title']} (k=4)", fontsize=28, fontweight='bold', ha='center', pad=20)    
    ax.set_xlabel('Effect Size Difference (Senescence - Cell Cycle)', fontsize=18)
    ax.set_ylabel('Cluster', fontsize=18)
    
    # Extended limit to +/- 12
    ax.set_xlim(-12, 12)
    
    # Dynamically scale y-axis based on max clusters
    max_cluster = max(clusters_list) if clusters_list else 8
    ax.set_ylim(0.5, max_cluster + 0.5)
    
    ax.set_xticks(range(-10, 11, 2))
    ax.set_xticklabels([r'$\leq -10$' if x <= -10 else (r'$\geq 10$' if x >= 10 else str(x)) for x in range(-10, 11, 2)], fontsize=14)
    
    ax.set_yticks(range(1, max_cluster + 1))
    ax.set_yticklabels([f'Cluster {c}' for c in range(1, max_cluster + 1)], fontsize=14)
    
    ax.axvline(x=0, color='grey', linewidth=0.8)
    ax.grid(True, linestyle='--', alpha=0.5, axis='x')
    
    # Allow the plot to stretch automatically to fit the taller figsize
    ax.set_aspect('auto') 
    
    plt.tight_layout()
    
    # Save the individual plot
    safe_filename = group['title'].replace(' ', '_')
    save_path = os.path.join(data_dir, f"{safe_filename}_zscore_diff_plot.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
    
    print(f"Saved {group['title']} plot to {save_path}")
    
    # Close the figure to free up memory before the next loop iteration
    plt.close(fig)

print("\nAll individual plots and standalone legend successfully generated and saved!")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os

# Define the path to your new single CSV file
csv_file_path = '/path/to/xGATE/data/senescence_4/ETO_CTRL_Competitive_full_dataset_8_clusters.csv'


# Map the CSV's raw 'Group' names to the clean titles you want displayed and saved
group_name_mapping = {
    'Ctrl': 'CTRL',
    'ETO Day 0': 'ETO 0',
    'ETO Day 1': 'ETO 1',
    'ETO Day 2': 'ETO 2',
    'ETO Day 4': 'ETO 4',
    'ETO Day 7': 'ETO 7',
    'ETO Day 10': 'ETO 10'
}

def determine_activity_color(emp_cc, emp_sen, comp_cc, comp_sen):
    """
    Assign color based on statistical significance (empirical and competitive p-values).
    green = Cell Cycle, blue = Senescence, grey = Neither/Not Significant
    """
    # Fallback for empty sparse clusters
    if pd.isna(emp_cc) or pd.isna(emp_sen):
        return 'grey'
        
    # Case 1: Neither is significant
    if emp_cc >= 0.05 and emp_sen >= 0.05:
        return 'grey'
        
    # Case 2: Only Cell Cycle is significant
    elif emp_cc < 0.05 and emp_sen >= 0.05:
        return 'lightgreen'
        
    # Case 3: Only Senescence is significant
    elif emp_sen < 0.05 and emp_cc >= 0.05:
        return 'skyblue'
        
    # Case 4: Both are significant (Empirical p-vals < 0.05)
    elif emp_cc < 0.05 and emp_sen < 0.05:
        # Check competitive p-values to break the tie
        if comp_cc < 0.05:
            return 'lightgreen'
        elif comp_sen < 0.05:
            return 'skyblue'
        else:
            return 'grey' # Failsafe if neither competitive p-val is < 0.05
            
    return 'grey' # Ultimate fallback

# =========================
# Standalone Legend Generation
# =========================
print("Generating standalone legend...")
legend_patches = [
    mpatches.Patch(color='skyblue', label='Cellular Senescence'),
    mpatches.Patch(color='lightgreen', label='Cell Cycle'),
    mpatches.Patch(color='grey', label='Not Significant')
]

fig_leg = plt.figure(figsize=(10, 2), facecolor='white')
ax_leg = fig_leg.add_subplot(111)
ax_leg.axis('off') # Hide axes for the legend image
ax_leg.legend(handles=legend_patches, loc='center', ncol=3, frameon=False, fontsize=18)

# Saving legend with the new 8_clusters tag
legend_save_path = os.path.join(data_dir, "Standalone_Legend_8_clusters.png")
fig_leg.savefig(legend_save_path, dpi=300, bbox_inches='tight', facecolor='white')
plt.close(fig_leg)
print(f"Legend saved to {legend_save_path}")

# =========================
# Individual Plot Generation
# =========================
print(f"\nLoading data from {csv_file_path}...")
df_full = pd.read_csv(csv_file_path)

# Iterate through each unique group in the dataframe
for raw_group_name in df_full['Group'].unique():
    
    # Get the clean title for the plot and saving (e.g. 'ETO Day 0' -> 'ETO 0')
    clean_title = group_name_mapping.get(raw_group_name, raw_group_name)
    print(f"Processing and plotting {clean_title}...")
    
    # Isolate data for just this group
    df_group = df_full[df_full['Group'] == raw_group_name].copy()
    
    # Find the largest cluster IN THIS GROUP to act as our baseline for maximum bubble size
    max_cells = df_group['Number of cells'].max()
    
    # Set the area of the absolute largest bubble
    max_bubble_area = 2500 
    
    clusters_list = df_group['Cluster'].tolist()
    z_diff_list = []
    colors = []
    scatter_sizes = []
    
    for index, row in df_group.iterrows():
        z_diff = row['Z-Score Difference (Senescence - Cell Cycle)']
        num_cells = row['Number of cells']
        
        # Extract the p-values for the color logic
        emp_cc = row['Cell Cycle Empirical P-value']
        emp_sen = row['Senescence Empirical P-value']
        comp_cc = row['Competitive P-value Cell Cycle']
        comp_sen = row['Competitive P-value Senescence']
        
        # Handle potential NaNs from sparse clusters
        if pd.isna(z_diff):
            z_diff = 0
            
        # Cap the z_diff to ±10
        z_diff_capped = 10 if z_diff >= 10 else (-10 if z_diff <= -10 else z_diff)
        z_diff_list.append(z_diff_capped)
        
        # Apply the p-value based color logic
        bubble_color = determine_activity_color(emp_cc, emp_sen, comp_cc, comp_sen)
        colors.append(bubble_color)
        
        # STRICT PROPORTIONALITY
        if max_cells > 0:
            proportional_area = (num_cells / max_cells) * max_bubble_area
        else:
            proportional_area = 0
            
        scatter_sizes.append(proportional_area)
        
    # Create the figure
    fig, ax = plt.subplots(figsize=(8, 6), facecolor='white')
    
    ax.scatter(
        z_diff_list, 
        clusters_list, 
        s=scatter_sizes, 
        c=colors, 
        alpha=0.6, 
        edgecolors='black'
    )
    
    # Set the title dynamically with the space included
    # Use clean_title instead of the leftover 'group' variable
    ax.set_title(f"{clean_title} (k=8)", fontsize=28, fontweight='bold', ha='center', pad=20)    
    ax.set_xlabel('Effect Size Difference (Senescence - Cell Cycle)', fontsize=18)
    ax.set_ylabel('Cluster', fontsize=18)
    
    ax.set_xlim(-12, 12)
    
    # Dynamically scale y-axis based on max clusters (should be 8 based on the new dataset)
    max_cluster = max(clusters_list) if clusters_list else 8
    ax.set_ylim(0.5, max_cluster + 0.5)
    
    ax.set_xticks(range(-10, 11, 2))
    ax.set_xticklabels([r'$\leq -10$' if x <= -10 else (r'$\geq 10$' if x >= 10 else str(x)) for x in range(-10, 11, 2)], fontsize=14)
    
    ax.set_yticks(range(1, max_cluster + 1))
    ax.set_yticklabels([f'Cluster {c}' for c in range(1, max_cluster + 1)], fontsize=14)
    
    ax.axvline(x=0, color='grey', linewidth=0.8)
    ax.grid(True, linestyle='--', alpha=0.5, axis='x')
    
    # Allow the plot to stretch automatically to fit the taller figsize
    ax.set_aspect('auto') 
    
    plt.tight_layout()
    
    # Save the individual plot with the "8_clusters" suffix
    # e.g., 'ETO_10_zscore_diff_plot_8_clusters.png'
    safe_filename = clean_title.replace(' ', '_')
    save_path = os.path.join(data_dir, f"/path/to/xGATE/data/senescence_4/{safe_filename}_zscore_diff_plot_8_clusters.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
    
    print(f"Saved {clean_title} plot to {save_path}")
    
    # Close the figure to free up memory before the next loop iteration
    plt.close(fig)

print("\nAll individual plots successfully generated and saved without overwriting original files!")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os

# Define the directory where your CSVs are saved
data_dir = '/path/to/xGATE/data/senescence_6/'

# Map the display titles to their corresponding file names (Titles include spaces)
file_mapping = [
    {'title': 'CTRL',   'filename': 'competitive_analysis_summary_ctrl.csv'},
    {'title': 'ETO 0',  'filename': 'competitive_analysis_summary_eto_0.csv'},
    {'title': 'ETO 1',  'filename': 'competitive_analysis_summary_eto_1.csv'},
    {'title': 'ETO 2',  'filename': 'competitive_analysis_summary_eto_2.csv'},
    {'title': 'ETO 4',  'filename': 'competitive_analysis_summary_eto_4.csv'},
    {'title': 'ETO 7',  'filename': 'competitive_analysis_summary_eto_7.csv'},
    {'title': 'ETO 10', 'filename': 'competitive_analysis_summary_eto_10.csv'}
]

def determine_activity_color(emp_cc, emp_sen, comp_cc, comp_sen):
    """
    Assign color based on statistical significance (empirical and competitive p-values).
    green = Cell Cycle, blue = Senescence, grey = Neither/Not Significant
    """
    # Fallback for empty sparse clusters
    if pd.isna(emp_cc) or pd.isna(emp_sen):
        return 'grey'
        
    # Case 1: Neither is significant
    if emp_cc >= 0.05 and emp_sen >= 0.05:
        return 'grey'
        
    # Case 2: Only Cell Cycle is significant
    elif emp_cc < 0.05 and emp_sen >= 0.05:
        return 'lightgreen'
        
    # Case 3: Only Senescence is significant
    elif emp_sen < 0.05 and emp_cc >= 0.05:
        return 'skyblue'
        
    # Case 4: Both are significant (Empirical p-vals < 0.05)
    elif emp_cc < 0.05 and emp_sen < 0.05:
        # Check competitive p-values to break the tie
        if comp_cc < 0.05:
            return 'lightgreen'
        elif comp_sen < 0.05:
            return 'skyblue'
        else:
            return 'grey' # Failsafe if neither competitive p-val is < 0.05
            
    return 'grey' # Ultimate fallback

# =========================
# Standalone Legend Generation
# =========================
print("Generating standalone legend...")
legend_patches = [
    mpatches.Patch(color='skyblue', label='Cellular Senescence'),
    mpatches.Patch(color='lightgreen', label='Cell Cycle'),
    mpatches.Patch(color='grey', label='Not Significant')  # Added grey patch
]

fig_leg = plt.figure(figsize=(10, 2), facecolor='white') # Made slightly wider to fit 3 items
ax_leg = fig_leg.add_subplot(111)
ax_leg.axis('off') # Hide axes for the legend image
ax_leg.legend(handles=legend_patches, loc='center', ncol=3, frameon=False, fontsize=18)

legend_save_path = os.path.join(data_dir, "Standalone_Legend.png")
fig_leg.savefig(legend_save_path, dpi=300, bbox_inches='tight', facecolor='white')
plt.close(fig_leg)
print(f"Legend saved to {legend_save_path}")


# =========================
# Individual Plot Generation
# =========================
print("\nLoading data and generating individual plots...")

for group in file_mapping:
    filepath = os.path.join(data_dir, group['filename'])
    
    if not os.path.exists(filepath):
        print(f"Warning: File {filepath} not found. Skipping {group['title']}.")
        continue
        
    print(f"Processing and plotting {group['title']}...")
    df = pd.read_csv(filepath)
    
    # Find the largest cluster in this dataset to act as our baseline for maximum bubble size
    max_cells = df['Number of cells'].max()
    
    # Set the area of the absolute largest bubble
    max_bubble_area = 2500 
    
    clusters_list = df['Cluster'].tolist()
    z_diff_list = []
    colors = []
    scatter_sizes = []
    
    for index, row in df.iterrows():
        z_diff = row['Z-Score Difference (Senescence - Cell Cycle)']
        num_cells = row['Number of cells']
        
        # Extract the p-values for the color logic
        emp_cc = row['Cell Cycle Empirical P-value']
        emp_sen = row['Senescence Empirical P-value']
        comp_cc = row['Competitive P-value Cell Cycle']
        comp_sen = row['Competitive P-value Senescence']
        
        # Handle potential NaNs from sparse clusters
        if pd.isna(z_diff):
            z_diff = 0
            
        # Cap the z_diff to ±10
        z_diff_capped = 10 if z_diff >= 10 else (-10 if z_diff <= -10 else z_diff)
        
        z_diff_list.append(z_diff_capped)
        
        # Apply the new p-value based color logic
        bubble_color = determine_activity_color(emp_cc, emp_sen, comp_cc, comp_sen)
        colors.append(bubble_color)
        
        # STRICT PROPORTIONALITY: 
        # Map the cell count linearly to the area ('s') based on the largest cluster.
        if max_cells > 0:
            proportional_area = (num_cells / max_cells) * max_bubble_area
        else:
            proportional_area = 0
            
        scatter_sizes.append(proportional_area)
        
    # Create the figure with your updated dimensions
    fig, ax = plt.subplots(figsize=(8, 6), facecolor='white')
    
    ax.scatter(
        z_diff_list, 
        clusters_list, 
        s=scatter_sizes, 
        c=colors, 
        alpha=0.6, 
        edgecolors='black'
    )
    
    # Set the title dynamically with the space included
    ax.set_title(f"{group['title']} (k=6)", fontsize=28, fontweight='bold', ha='center', pad=20)    
    ax.set_xlabel('Effect Size Difference (Senescence - Cell Cycle)', fontsize=18)
    ax.set_ylabel('Cluster', fontsize=18)
    
    # Extended limit to +/- 12
    ax.set_xlim(-12, 12)
    
    # Dynamically scale y-axis based on max clusters
    max_cluster = max(clusters_list) if clusters_list else 8
    ax.set_ylim(0.5, max_cluster + 0.5)
    
    ax.set_xticks(range(-10, 11, 2))
    ax.set_xticklabels([r'$\leq -10$' if x <= -10 else (r'$\geq 10$' if x >= 10 else str(x)) for x in range(-10, 11, 2)], fontsize=14)
    
    ax.set_yticks(range(1, max_cluster + 1))
    ax.set_yticklabels([f'Cluster {c}' for c in range(1, max_cluster + 1)], fontsize=14)
    
    ax.axvline(x=0, color='grey', linewidth=0.8)
    ax.grid(True, linestyle='--', alpha=0.5, axis='x')
    
    # Allow the plot to stretch automatically to fit the taller figsize
    ax.set_aspect('auto') 
    
    plt.tight_layout()
    
    # Save the individual plot
    safe_filename = group['title'].replace(' ', '_')
    save_path = os.path.join(data_dir, f"{safe_filename}_zscore_diff_plot.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
    
    print(f"Saved {group['title']} plot to {save_path}")
    
    # Close the figure to free up memory before the next loop iteration
    plt.close(fig)

print("\nAll individual plots and standalone legend successfully generated and saved!")

In [ ]:
# k = 6 plot with percentages attached

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os
import math  # Added to calculate bubble radius for text offset

# Define the directory where your CSVs are saved
data_dir = '/path/to/xGATE/data/senescence_6/'

# Map the display titles to their corresponding file names (Titles include spaces)
file_mapping = [
    {'title': 'CTRL',   'filename': 'competitive_analysis_summary_ctrl_2.csv'},
    {'title': 'ETO 0',  'filename': 'competitive_analysis_summary_eto_0.csv'},
    {'title': 'ETO 1',  'filename': 'competitive_analysis_summary_eto_1.csv'},
    {'title': 'ETO 2',  'filename': 'competitive_analysis_summary_eto_2.csv'},
    {'title': 'ETO 4',  'filename': 'competitive_analysis_summary_eto_4.csv'},
    {'title': 'ETO 7',  'filename': 'competitive_analysis_summary_eto_7.csv'},
    {'title': 'ETO 10', 'filename': 'competitive_analysis_summary_eto_10.csv'}
]


def determine_activity_color(emp_cc, emp_sen, comp_cc, comp_sen):
    """
    Assign color based on statistical significance (empirical and competitive p-values).
    green = Cell Cycle, blue = Senescence, grey = Neither/Not Significant
    """
    # Fallback for empty sparse clusters
    if pd.isna(emp_cc) or pd.isna(emp_sen):
        return 'grey'
        
    # Case 1: Neither is significant
    if emp_cc >= 0.05 and emp_sen >= 0.05:
        return 'grey'
        
    # Case 2: Only Cell Cycle is significant
    elif emp_cc < 0.05 and emp_sen >= 0.05:
        return 'lightgreen'
        
    # Case 3: Only Senescence is significant
    elif emp_sen < 0.05 and emp_cc >= 0.05:
        return 'skyblue'
        
    # Case 4: Both are significant (Empirical p-vals < 0.05)
    elif emp_cc < 0.05 and emp_sen < 0.05:
        # Check competitive p-values to break the tie
        if comp_cc < 0.05:
            return 'lightgreen'
        elif comp_sen < 0.05:
            return 'skyblue'
        else:
            return 'grey' # Failsafe if neither competitive p-val is < 0.05
            
    return 'grey' # Ultimate fallback

# =========================
# Standalone Legend Generation
# =========================
print("Generating standalone legend...")
legend_patches = [
    mpatches.Patch(color='skyblue', label='Cellular Senescence'),
    mpatches.Patch(color='lightgreen', label='Cell Cycle'),
    mpatches.Patch(color='grey', label='Not Significant')  # Added grey patch
]

fig_leg = plt.figure(figsize=(10, 2), facecolor='white') 
ax_leg = fig_leg.add_subplot(111)
ax_leg.axis('off') 
ax_leg.legend(handles=legend_patches, loc='center', ncol=3, frameon=False, fontsize=18)

legend_save_path = os.path.join(data_dir, "Standalone_Legend.png")
fig_leg.savefig(legend_save_path, dpi=300, bbox_inches='tight', facecolor='white')
plt.close(fig_leg)
print(f"Legend saved to {legend_save_path}")


# =========================
# Individual Plot Generation
# =========================
print("\nLoading data and generating individual plots...")

for group in file_mapping:
    filepath = os.path.join(data_dir, group['filename'])
    
    if not os.path.exists(filepath):
        print(f"Warning: File {filepath} not found. Skipping {group['title']}.")
        continue
        
    print(f"Processing and plotting {group['title']}...")
    df = pd.read_csv(filepath)
    
    # Calculate total cells for percentage math
    total_cells_in_dataset = df['Number of cells'].sum()
    
    # Find the largest cluster in this dataset to act as our baseline for maximum bubble size
    max_cells = df['Number of cells'].max()
    
    # Set the area of the absolute largest bubble
    max_bubble_area = 2500 
    
    clusters_list = df['Cluster'].tolist()
    z_diff_list = []
    colors = []
    scatter_sizes = []
    
    # Create the figure here so we can annotate it during the loop
    fig, ax = plt.subplots(figsize=(6, 6), facecolor='white')
    
    for index, row in df.iterrows():
        z_diff = row['Z-Score Difference (Senescence - Cell Cycle)']
        num_cells = row['Number of cells']
        cluster_val = row['Cluster']
        
        # Extract the p-values for the color logic
        emp_cc = row['Cell Cycle Empirical P-value']
        emp_sen = row['Senescence Empirical P-value']
        comp_cc = row['Competitive P-value Cell Cycle']
        comp_sen = row['Competitive P-value Senescence']
        
        # Handle potential NaNs from sparse clusters
        if pd.isna(z_diff):
            z_diff = 0
            
        # Cap the z_diff to ±10
        z_diff_capped = 10 if z_diff >= 10 else (-10 if z_diff <= -10 else z_diff)
        
        z_diff_list.append(z_diff_capped)
        
        # Apply the new p-value based color logic
        bubble_color = determine_activity_color(emp_cc, emp_sen, comp_cc, comp_sen)
        colors.append(bubble_color)
        
        # STRICT PROPORTIONALITY: 
        # Map the cell count linearly to the area ('s') based on the largest cluster.
        if max_cells > 0:
            proportional_area = (num_cells / max_cells) * max_bubble_area
        else:
            proportional_area = 0
            
        scatter_sizes.append(proportional_area)
        
        # --- UPDATED ANNOTATION LOGIC ---
        if total_cells_in_dataset > 0:
            perc = (num_cells / total_cells_in_dataset) * 100
            perc_text = f"{perc:.1f}%"
            
            # Matplotlib scatter 's' is area in points^2. Radius in points is sqrt(area / pi)
            radius_pts = math.sqrt(proportional_area / math.pi)
            padding_pts = 6 # Extra buffer space between the circle and the text
            
            if z_diff_capped < 0:
                # Text goes to the RIGHT for negative values
                x_offset = radius_pts + padding_pts
                ha = 'left'
            else:
                # Text goes to the LEFT for positive values (and 0)
                x_offset = -(radius_pts + padding_pts)
                ha = 'right'
                
            ax.annotate(
                perc_text,
                xy=(z_diff_capped, cluster_val),
                xytext=(x_offset, 0),
                textcoords='offset points', # This ignores axes scale and uses fixed point sizes
                ha=ha,
                va='center',
                fontsize=11,
                color='black',
                fontweight='medium'
            )
        
    ax.scatter(
        z_diff_list, 
        clusters_list, 
        s=scatter_sizes, 
        c=colors, 
        alpha=0.6, 
        edgecolors='black'
    )
    
    # Set the title dynamically with the space included
    ax.set_title(f"{group['title']} (k=6)", fontsize=28, fontweight='bold', ha='center', pad=20)   
    
    # Increased font sizes for labels
    ax.set_xlabel('Effect Size Difference', fontsize=22)
    ax.set_ylabel('Cluster', fontsize=22)
    
    # Extended limit to +/- 12
    ax.set_xlim(-12, 12)
    
    # Dynamically scale y-axis based on max clusters
    max_cluster = max(clusters_list) if clusters_list else 8
    ax.set_ylim(0.5, max_cluster + 0.5)
    
    ax.set_xticks(range(-10, 11, 5)) # Changed step from 2 to 5
    ax.set_xticklabels([r'$\leq -10$' if x <= -10 else (r'$\geq 10$' if x >= 10 else str(x)) for x in range(-10, 11, 5)], fontsize=14)    
    ax.set_yticks(range(1, max_cluster + 1))
    ax.set_yticklabels([f'Cluster {c}' for c in range(1, max_cluster + 1)], fontsize=12)
    
    ax.axvline(x=0, color='grey', linewidth=0.8)
    ax.grid(True, linestyle='--', alpha=0.5, axis='x')
    
    # Allow the plot to stretch automatically to fit the taller figsize
    ax.set_aspect('auto') 
    
    plt.tight_layout()
    
    # Save the individual plot
    safe_filename = group['title'].replace(' ', '_')
    save_path = os.path.join(data_dir, f"{safe_filename}_zscore_diff_plot.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
    
    print(f"Saved {group['title']} plot to {save_path}")
    
    # Close the figure to free up memory before the next loop iteration
    plt.close(fig)

print("\nAll individual plots and standalone legend successfully generated and saved!")

In [ ]:
# k = 4 with percentages 
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os
import math  # Added to calculate bubble radius for text offset

# Define the directory where your CSVs are saved
data_dir = '/path/to/xGATE/data/senescence_4/'

# Map the display titles to their corresponding file names (Titles include spaces)
file_mapping = [
    {'title': 'CTRL',   'filename': 'competitive_analysis_summary_ctrl.csv'},
    {'title': 'ETO 0',  'filename': 'competitive_analysis_summary_eto_0.csv'},
    {'title': 'ETO 1',  'filename': 'competitive_analysis_summary_eto_1.csv'},
    {'title': 'ETO 2',  'filename': 'competitive_analysis_summary_eto_2.csv'},
    {'title': 'ETO 4',  'filename': 'competitive_analysis_summary_eto_4.csv'},
    {'title': 'ETO 7',  'filename': 'competitive_analysis_summary_eto_7.csv'},
    {'title': 'ETO 10', 'filename': 'competitive_analysis_summary_eto_10.csv'}
]


def determine_activity_color(emp_cc, emp_sen, comp_cc, comp_sen):
    """
    Assign color based on statistical significance (empirical and competitive p-values).
    green = Cell Cycle, blue = Senescence, grey = Neither/Not Significant
    """
    # Fallback for empty sparse clusters
    if pd.isna(emp_cc) or pd.isna(emp_sen):
        return 'grey'
        
    # Case 1: Neither is significant
    if emp_cc >= 0.05 and emp_sen >= 0.05:
        return 'grey'
        
    # Case 2: Only Cell Cycle is significant
    elif emp_cc < 0.05 and emp_sen >= 0.05:
        return 'lightgreen'
        
    # Case 3: Only Senescence is significant
    elif emp_sen < 0.05 and emp_cc >= 0.05:
        return 'skyblue'
        
    # Case 4: Both are significant (Empirical p-vals < 0.05)
    elif emp_cc < 0.05 and emp_sen < 0.05:
        # Check competitive p-values to break the tie
        if comp_cc < 0.05:
            return 'lightgreen'
        elif comp_sen < 0.05:
            return 'skyblue'
        else:
            return 'grey' # Failsafe if neither competitive p-val is < 0.05
            
    return 'grey' # Ultimate fallback

# =========================
# Standalone Legend Generation
# =========================
print("Generating standalone legend...")
legend_patches = [
    mpatches.Patch(color='skyblue', label='Cellular Senescence'),
    mpatches.Patch(color='lightgreen', label='Cell Cycle'),
    mpatches.Patch(color='grey', label='Not Significant')  # Added grey patch
]

fig_leg = plt.figure(figsize=(10, 2), facecolor='white') 
ax_leg = fig_leg.add_subplot(111)
ax_leg.axis('off') 
ax_leg.legend(handles=legend_patches, loc='center', ncol=3, frameon=False, fontsize=18)

legend_save_path = os.path.join(data_dir, "Standalone_Legend.png")
fig_leg.savefig(legend_save_path, dpi=300, bbox_inches='tight', facecolor='white')
plt.close(fig_leg)
print(f"Legend saved to {legend_save_path}")


# =========================
# Individual Plot Generation
# =========================
print("\nLoading data and generating individual plots...")

for group in file_mapping:
    filepath = os.path.join(data_dir, group['filename'])
    
    if not os.path.exists(filepath):
        print(f"Warning: File {filepath} not found. Skipping {group['title']}.")
        continue
        
    print(f"Processing and plotting {group['title']}...")
    df = pd.read_csv(filepath)
    
    # Calculate total cells for percentage math
    total_cells_in_dataset = df['Number of cells'].sum()
    
    # Find the largest cluster in this dataset to act as our baseline for maximum bubble size
    max_cells = df['Number of cells'].max()
    
    # Set the area of the absolute largest bubble
    max_bubble_area = 2500 
    
    clusters_list = df['Cluster'].tolist()
    z_diff_list = []
    colors = []
    scatter_sizes = []
    
    # Create the figure here so we can annotate it during the loop
    fig, ax = plt.subplots(figsize=(6, 6), facecolor='white')
    
    for index, row in df.iterrows():
        z_diff = row['Z-Score Difference (Senescence - Cell Cycle)']
        num_cells = row['Number of cells']
        cluster_val = row['Cluster']
        
        # Extract the p-values for the color logic
        emp_cc = row['Cell Cycle Empirical P-value']
        emp_sen = row['Senescence Empirical P-value']
        comp_cc = row['Competitive P-value Cell Cycle']
        comp_sen = row['Competitive P-value Senescence']
        
        # Handle potential NaNs from sparse clusters
        if pd.isna(z_diff):
            z_diff = 0
            
        # Cap the z_diff to ±10
        z_diff_capped = 10 if z_diff >= 10 else (-10 if z_diff <= -10 else z_diff)
        
        z_diff_list.append(z_diff_capped)
        
        # Apply the new p-value based color logic
        bubble_color = determine_activity_color(emp_cc, emp_sen, comp_cc, comp_sen)
        colors.append(bubble_color)
        
        # STRICT PROPORTIONALITY: 
        # Map the cell count linearly to the area ('s') based on the largest cluster.
        if max_cells > 0:
            proportional_area = (num_cells / max_cells) * max_bubble_area
        else:
            proportional_area = 0
            
        scatter_sizes.append(proportional_area)
        
        # --- UPDATED ANNOTATION LOGIC ---
        if total_cells_in_dataset > 0:
            perc = (num_cells / total_cells_in_dataset) * 100
            perc_text = f"{perc:.1f}%"
            
            # Matplotlib scatter 's' is area in points^2. Radius in points is sqrt(area / pi)
            radius_pts = math.sqrt(proportional_area / math.pi)
            padding_pts = 6 # Extra buffer space between the circle and the text
            
            if z_diff_capped < 0:
                # Text goes to the RIGHT for negative values
                x_offset = radius_pts + padding_pts
                ha = 'left'
            else:
                # Text goes to the LEFT for positive values (and 0)
                x_offset = -(radius_pts + padding_pts)
                ha = 'right'
                
            ax.annotate(
                perc_text,
                xy=(z_diff_capped, cluster_val),
                xytext=(x_offset, 0),
                textcoords='offset points', # This ignores axes scale and uses fixed point sizes
                ha=ha,
                va='center',
                fontsize=11,
                color='black',
                fontweight='medium'
            )
        
    ax.scatter(
        z_diff_list, 
        clusters_list, 
        s=scatter_sizes, 
        c=colors, 
        alpha=0.6, 
        edgecolors='black'
    )
    
    # Set the title dynamically with the space included
    ax.set_title(f"{group['title']} (k=4)", fontsize=28, fontweight='bold', ha='center', pad=20)   
    
    # Increased font sizes for labels
    ax.set_xlabel('Effect Size Difference', fontsize=22)
    ax.set_ylabel('Cluster', fontsize=22)
    
    # Extended limit to +/- 12
    ax.set_xlim(-12, 12)
    
    # Dynamically scale y-axis based on max clusters
    max_cluster = max(clusters_list) if clusters_list else 8
    ax.set_ylim(0.5, max_cluster + 0.5)
    
    ax.set_xticks(range(-10, 11, 5)) # Changed step from 2 to 5
    ax.set_xticklabels([r'$\leq -10$' if x <= -10 else (r'$\geq 10$' if x >= 10 else str(x)) for x in range(-10, 11, 5)], fontsize=14)    
    ax.set_yticks(range(1, max_cluster + 1))
    ax.set_yticklabels([f'Cluster {c}' for c in range(1, max_cluster + 1)], fontsize=12)
    
    ax.axvline(x=0, color='grey', linewidth=0.8)
    ax.grid(True, linestyle='--', alpha=0.5, axis='x')
    
    # Allow the plot to stretch automatically to fit the taller figsize
    ax.set_aspect('auto') 
    
    plt.tight_layout()
    
    # Save the individual plot
    safe_filename = group['title'].replace(' ', '_')
    save_path = os.path.join(data_dir, f"{safe_filename}_zscore_diff_plot.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
    
    print(f"Saved {group['title']} plot to {save_path}")
    
    # Close the figure to free up memory before the next loop iteration
    plt.close(fig)

print("\nAll individual plots and standalone legend successfully generated and saved!")

# Top code works fine, this bottom code is to align all of the ctrl & eto groups in 1 row in the plot

In [ ]:
# k = 4 with percentages 
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os
import math

# Define the directory where your CSVs are saved
data_dir = '/path/to/xGATE/data/senescence_4/'

# Map the display titles to their corresponding file names
file_mapping = [
    {'title': 'CTRL',   'filename': 'competitive_analysis_summary_ctrl.csv'},
    {'title': 'ETO 0',  'filename': 'competitive_analysis_summary_eto_0.csv'},
    {'title': 'ETO 1',  'filename': 'competitive_analysis_summary_eto_1.csv'},
    {'title': 'ETO 2',  'filename': 'competitive_analysis_summary_eto_2.csv'},
    {'title': 'ETO 4',  'filename': 'competitive_analysis_summary_eto_4.csv'},
    {'title': 'ETO 7',  'filename': 'competitive_analysis_summary_eto_7.csv'},
    {'title': 'ETO 10', 'filename': 'competitive_analysis_summary_eto_10.csv'}
]

def determine_activity_color(emp_cc, emp_sen, comp_cc, comp_sen):
    if pd.isna(emp_cc) or pd.isna(emp_sen):
        return 'grey'
    if emp_cc >= 0.05 and emp_sen >= 0.05:
        return 'grey'
    elif emp_cc < 0.05 and emp_sen >= 0.05:
        return 'lightgreen'
    elif emp_sen < 0.05 and emp_cc >= 0.05:
        return 'skyblue'
    elif emp_cc < 0.05 and emp_sen < 0.05:
        if comp_cc < 0.05:
            return 'lightgreen'
        elif comp_sen < 0.05:
            return 'skyblue'
        else:
            return 'grey'
    return 'grey'

# =========================
# Standalone Legend Generation
# =========================
print("Generating standalone legend...")
legend_patches = [
    mpatches.Patch(color='skyblue', label='Cellular Senescence'),
    mpatches.Patch(color='lightgreen', label='Cell Cycle'),
    mpatches.Patch(color='grey', label='Not Significant')
]

fig_leg = plt.figure(figsize=(10, 2), facecolor='white') 
ax_leg = fig_leg.add_subplot(111)
ax_leg.axis('off') 
ax_leg.legend(handles=legend_patches, loc='center', ncol=3, frameon=False, fontsize=18)

legend_save_path = os.path.join(data_dir, "Standalone_Legend.png")
fig_leg.savefig(legend_save_path, dpi=300, bbox_inches='tight', facecolor='white')
plt.close(fig_leg)
print(f"Legend saved to {legend_save_path}")

# =========================
# Combined Plot Generation
# =========================
print("\nLoading data and generating combined plot...")

# First, filter out any files that don't exist and find the global max cluster for the shared Y-axis
valid_groups = []
global_max_cluster = 0

for group in file_mapping:
    filepath = os.path.join(data_dir, group['filename'])
    if os.path.exists(filepath):
        valid_groups.append(group)
        df = pd.read_csv(filepath)
        curr_max = df['Cluster'].max()
        if curr_max > global_max_cluster:
            global_max_cluster = curr_max
    else:
        print(f"Warning: File {filepath} not found. Skipping {group['title']}.")

# Fallback if empty
if pd.isna(global_max_cluster) or global_max_cluster == 0:
    global_max_cluster = 4

num_plots = len(valid_groups)

# Create a figure with 1 row and 'num_plots' columns. 
# Width is roughly 5 inches per plot, height is 6 inches.
fig, axes = plt.subplots(nrows=1, ncols=num_plots, figsize=(5 * num_plots, 6), sharey=True, facecolor='white')

# Ensure axes is iterable even if there's only 1 valid file
if num_plots == 1:
    axes = [axes]

for i, group in enumerate(valid_groups):
    ax = axes[i]
    filepath = os.path.join(data_dir, group['filename'])
    df = pd.read_csv(filepath)
    
    total_cells_in_dataset = df['Number of cells'].sum()
    max_cells = df['Number of cells'].max()
    max_bubble_area = 2500 
    
    clusters_list = df['Cluster'].tolist()
    z_diff_list = []
    colors = []
    scatter_sizes = []
    
    for index, row in df.iterrows():
        z_diff = row['Z-Score Difference (Senescence - Cell Cycle)']
        num_cells = row['Number of cells']
        cluster_val = row['Cluster']
        
        emp_cc = row['Cell Cycle Empirical P-value']
        emp_sen = row['Senescence Empirical P-value']
        comp_cc = row['Competitive P-value Cell Cycle']
        comp_sen = row['Competitive P-value Senescence']
        
        if pd.isna(z_diff):
            z_diff = 0
            
        z_diff_capped = 10 if z_diff >= 10 else (-10 if z_diff <= -10 else z_diff)
        z_diff_list.append(z_diff_capped)
        
        bubble_color = determine_activity_color(emp_cc, emp_sen, comp_cc, comp_sen)
        colors.append(bubble_color)
        
        if max_cells > 0:
            proportional_area = (num_cells / max_cells) * max_bubble_area
        else:
            proportional_area = 0
            
        scatter_sizes.append(proportional_area)
        
        # Annotation Logic
        if total_cells_in_dataset > 0:
            perc = (num_cells / total_cells_in_dataset) * 100
            perc_text = f"{perc:.1f}%"
            
            radius_pts = math.sqrt(proportional_area / math.pi)
            padding_pts = 6 
            
            if z_diff_capped < 0:
                x_offset = radius_pts + padding_pts
                ha = 'left'
            else:
                x_offset = -(radius_pts + padding_pts)
                ha = 'right'
                
            ax.annotate(
                perc_text,
                xy=(z_diff_capped, cluster_val),
                xytext=(x_offset, 0),
                textcoords='offset points', 
                ha=ha,
                va='center',
                fontsize=16,
                color='black',
                fontweight='medium'
            )
        
    ax.scatter(
        z_diff_list, 
        clusters_list, 
        s=scatter_sizes, 
        c=colors, 
        alpha=0.6, 
        edgecolors='black'
    )
    
    # --- FIXED TITLE HERE ---
    ax.set_title(f"{group['title']} (k=4)", fontsize=24, fontweight='bold', ha='center', pad=15)   
    ax.set_xlabel('Effect Size Difference', fontsize=20)
    
    # Only show the Y-axis label on the very first (left-most) plot
    if i == 0:
        ax.set_ylabel('', fontsize=22)
    
    ax.set_xlim(-12, 12)
    ax.set_ylim(0.5, global_max_cluster + 0.5)
    
    ax.set_xticks(range(-10, 11, 5)) 
    ax.set_xticklabels([r'$\leq -10$' if x <= -10 else (r'$\geq 10$' if x >= 10 else str(x)) for x in range(-10, 11, 5)], fontsize=16)   
    
    # We apply y-ticks to all so the grid aligns, but sharey=True hides the labels on subplots > 0
    ax.set_yticks(range(1, int(global_max_cluster) + 1))
    ax.set_yticklabels([f'Cluster {c}' for c in range(1, int(global_max_cluster) + 1)], fontsize=22, fontweight= 'bold')
    
    ax.axvline(x=0, color='grey', linewidth=0.8)
    ax.grid(True, linestyle='--', alpha=0.5, axis='x')
    
# Adjust spacing so the plots don't overlap
plt.tight_layout()

# Save the combined plot
save_path = os.path.join(data_dir, "Combined_Zscore_Diff_Plot_k4.png")
plt.savefig(save_path, dpi=400, bbox_inches='tight', facecolor='white')
print(f"Saved combined plot to {save_path}")

plt.close(fig)
print("\nCombined plot and standalone legend successfully generated and saved!")

In [ ]:
# k = 6 with percentages 
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os
import math

# Define the directory where your CSVs are saved
data_dir = '/path/to/xGATE/data/senescence_6/'

# Map the display titles to their corresponding file names
file_mapping = [
    {'title': 'CTRL',   'filename': 'competitive_analysis_summary_ctrl_2.csv'},
    {'title': 'ETO 0',  'filename': 'competitive_analysis_summary_eto_0.csv'},
    {'title': 'ETO 1',  'filename': 'competitive_analysis_summary_eto_1.csv'},
    {'title': 'ETO 2',  'filename': 'competitive_analysis_summary_eto_2.csv'},
    {'title': 'ETO 4',  'filename': 'competitive_analysis_summary_eto_4.csv'},
    {'title': 'ETO 7',  'filename': 'competitive_analysis_summary_eto_7.csv'},
    {'title': 'ETO 10', 'filename': 'competitive_analysis_summary_eto_10.csv'}
]

def determine_activity_color(emp_cc, emp_sen, comp_cc, comp_sen):
    if pd.isna(emp_cc) or pd.isna(emp_sen):
        return 'grey'
    if emp_cc >= 0.05 and emp_sen >= 0.05:
        return 'grey'
    elif emp_cc < 0.05 and emp_sen >= 0.05:
        return 'lightgreen'
    elif emp_sen < 0.05 and emp_cc >= 0.05:
        return 'skyblue'
    elif emp_cc < 0.05 and emp_sen < 0.05:
        if comp_cc < 0.05:
            return 'lightgreen'
        elif comp_sen < 0.05:
            return 'skyblue'
        else:
            return 'grey'
    return 'grey'

# =========================
# Standalone Legend Generation
# =========================
print("Generating standalone legend...")
legend_patches = [
    mpatches.Patch(color='skyblue', label='Cellular Senescence'),
    mpatches.Patch(color='lightgreen', label='Cell Cycle'),
    mpatches.Patch(color='grey', label='Not Significant')
]

fig_leg = plt.figure(figsize=(10, 2), facecolor='white') 
ax_leg = fig_leg.add_subplot(111)
ax_leg.axis('off') 
ax_leg.legend(handles=legend_patches, loc='center', ncol=3, frameon=False, fontsize=18)

legend_save_path = os.path.join(data_dir, "Standalone_Legend.png")
fig_leg.savefig(legend_save_path, dpi=300, bbox_inches='tight', facecolor='white')
plt.close(fig_leg)
print(f"Legend saved to {legend_save_path}")

# =========================
# Combined Plot Generation
# =========================
print("\nLoading data and generating combined plot...")

# First, filter out any files that don't exist and find the global max cluster for the shared Y-axis
valid_groups = []
global_max_cluster = 0

for group in file_mapping:
    filepath = os.path.join(data_dir, group['filename'])
    if os.path.exists(filepath):
        valid_groups.append(group)
        df = pd.read_csv(filepath)
        curr_max = df['Cluster'].max()
        if curr_max > global_max_cluster:
            global_max_cluster = curr_max
    else:
        print(f"Warning: File {filepath} not found. Skipping {group['title']}.")

# Fallback if empty
if pd.isna(global_max_cluster) or global_max_cluster == 0:
    global_max_cluster = 4

num_plots = len(valid_groups)

# Create a figure with 1 row and 'num_plots' columns. 
# Width is roughly 5 inches per plot, height is 6 inches.
fig, axes = plt.subplots(nrows=1, ncols=num_plots, figsize=(5 * num_plots, 6), sharey=True, facecolor='white')

# Ensure axes is iterable even if there's only 1 valid file
if num_plots == 1:
    axes = [axes]

for i, group in enumerate(valid_groups):
    ax = axes[i]
    filepath = os.path.join(data_dir, group['filename'])
    df = pd.read_csv(filepath)
    
    total_cells_in_dataset = df['Number of cells'].sum()
    max_cells = df['Number of cells'].max()
    max_bubble_area = 2500 
    
    clusters_list = df['Cluster'].tolist()
    z_diff_list = []
    colors = []
    scatter_sizes = []
    
    for index, row in df.iterrows():
        z_diff = row['Z-Score Difference (Senescence - Cell Cycle)']
        num_cells = row['Number of cells']
        cluster_val = row['Cluster']
        
        emp_cc = row['Cell Cycle Empirical P-value']
        emp_sen = row['Senescence Empirical P-value']
        comp_cc = row['Competitive P-value Cell Cycle']
        comp_sen = row['Competitive P-value Senescence']
        
        if pd.isna(z_diff):
            z_diff = 0
            
        z_diff_capped = 10 if z_diff >= 10 else (-10 if z_diff <= -10 else z_diff)
        z_diff_list.append(z_diff_capped)
        
        bubble_color = determine_activity_color(emp_cc, emp_sen, comp_cc, comp_sen)
        colors.append(bubble_color)
        
        if max_cells > 0:
            proportional_area = (num_cells / max_cells) * max_bubble_area
        else:
            proportional_area = 0
            
        scatter_sizes.append(proportional_area)
        
        # Annotation Logic
        if total_cells_in_dataset > 0:
            perc = (num_cells / total_cells_in_dataset) * 100
            perc_text = f"{perc:.1f}%"
            
            radius_pts = math.sqrt(proportional_area / math.pi)
            padding_pts = 6 
            
            if z_diff_capped < 0:
                x_offset = radius_pts + padding_pts
                ha = 'left'
            else:
                x_offset = -(radius_pts + padding_pts)
                ha = 'right'
                
            ax.annotate(
                perc_text,
                xy=(z_diff_capped, cluster_val),
                xytext=(x_offset, 0),
                textcoords='offset points', 
                ha=ha,
                va='center',
                fontsize=16,
                color='black',
                fontweight='medium'
            )
        
    ax.scatter(
        z_diff_list, 
        clusters_list, 
        s=scatter_sizes, 
        c=colors, 
        alpha=0.6, 
        edgecolors='black'
    )
    
    # --- FIXED TITLE HERE ---
    ax.set_title(f"{group['title']} (k=6)", fontsize=24, fontweight='bold', ha='center', pad=15)   
    ax.set_xlabel('Effect Size Difference', fontsize=20)
    
    # Only show the Y-axis label on the very first (left-most) plot
    if i == 0:
        ax.set_ylabel('', fontsize=22)
    
    ax.set_xlim(-12, 12)
    ax.set_ylim(0.5, global_max_cluster + 0.5)
    
    ax.set_xticks(range(-10, 11, 5)) 
    ax.set_xticklabels([r'$\leq -10$' if x <= -10 else (r'$\geq 10$' if x >= 10 else str(x)) for x in range(-10, 11, 5)], fontsize=16)   
    
    # We apply y-ticks to all so the grid aligns, but sharey=True hides the labels on subplots > 0
    ax.set_yticks(range(1, int(global_max_cluster) + 1))
    ax.set_yticklabels([f'Cluster {c}' for c in range(1, int(global_max_cluster) + 1)], fontsize=16)
    
    ax.axvline(x=0, color='grey', linewidth=0.8)
    ax.grid(True, linestyle='--', alpha=0.5, axis='x')
    
# Adjust spacing so the plots don't overlap
plt.tight_layout()

# Save the combined plot
save_path = os.path.join(data_dir, "Combined_Zscore_Diff_Plot_k6.png")
plt.savefig(save_path, dpi=400, bbox_inches='tight', facecolor='white')
print(f"Saved combined plot to {save_path}")

plt.close(fig)
print("\nCombined plot and standalone legend successfully generated and saved!")

In [ ]:
# k =8
# k = 8 with percentages in a Combined Subplot
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os
import math

# Define the paths
data_dir = '/path/to/xGATE/data/senescence_8/'
csv_file_path = os.path.join(data_dir, 'ETO_CTRL_Competitive_full_dataset_8_clusters.csv')

# Map the CSV's raw 'Group' names to the clean titles
# Using a dictionary here also safely defines the chronological order for the subplots
group_name_mapping = {
    'Ctrl': 'CTRL',
    'ETO Day 0': 'ETO 0',
    'ETO Day 1': 'ETO 1',
    'ETO Day 2': 'ETO 2',
    'ETO Day 4': 'ETO 4',
    'ETO Day 7': 'ETO 7',
    'ETO Day 10': 'ETO 10'
}

def determine_activity_color(emp_cc, emp_sen, comp_cc, comp_sen):
    if pd.isna(emp_cc) or pd.isna(emp_sen):
        return 'grey'
    if emp_cc >= 0.05 and emp_sen >= 0.05:
        return 'grey'
    elif emp_cc < 0.05 and emp_sen >= 0.05:
        return 'lightgreen'
    elif emp_sen < 0.05 and emp_cc >= 0.05:
        return 'skyblue'
    elif emp_cc < 0.05 and emp_sen < 0.05:
        if comp_cc < 0.05:
            return 'lightgreen'
        elif comp_sen < 0.05:
            return 'skyblue'
        else:
            return 'grey'
    return 'grey'

# =========================
# Standalone Legend Generation
# =========================
print("Generating standalone legend...")
legend_patches = [
    mpatches.Patch(color='skyblue', label='Cellular Senescence'),
    mpatches.Patch(color='lightgreen', label='Cell Cycle'),
    mpatches.Patch(color='grey', label='Not Significant')
]

fig_leg = plt.figure(figsize=(10, 2), facecolor='white')
ax_leg = fig_leg.add_subplot(111)
ax_leg.axis('off') 
ax_leg.legend(handles=legend_patches, loc='center', ncol=3, frameon=False, fontsize=18)

legend_save_path = os.path.join(data_dir, "Standalone_Legend_8_clusters.png")
fig_leg.savefig(legend_save_path, dpi=300, bbox_inches='tight', facecolor='white')
plt.close(fig_leg)
print(f"Legend saved to {legend_save_path}")

# =========================
# Combined Plot Generation
# =========================
print(f"\nLoading data from {csv_file_path}...")
df_full = pd.read_csv(csv_file_path)

# Filter groups to only those present in the CSV, keeping the explicit order of our mapping
valid_raw_groups = [g for g in group_name_mapping.keys() if g in df_full['Group'].unique()]

# Find the global max cluster for the shared Y-axis (fallback to 8)
global_max_cluster = df_full['Cluster'].max()
if pd.isna(global_max_cluster) or global_max_cluster == 0:
    global_max_cluster = 8

num_plots = len(valid_raw_groups)

# Create a figure with 1 row and 'num_plots' columns
fig, axes = plt.subplots(nrows=1, ncols=num_plots, figsize=(5 * num_plots, 6), sharey=True, facecolor='white')

# Ensure axes is iterable
if num_plots == 1:
    axes = [axes]

for i, raw_group_name in enumerate(valid_raw_groups):
    ax = axes[i]
    clean_title = group_name_mapping[raw_group_name]
    
    # Isolate data for just this group
    df_group = df_full[df_full['Group'] == raw_group_name].copy()
    
    total_cells_in_dataset = df_group['Number of cells'].sum()
    max_cells = df_group['Number of cells'].max()
    max_bubble_area = 2500 
    
    clusters_list = df_group['Cluster'].tolist()
    z_diff_list = []
    colors = []
    scatter_sizes = []
    
    for index, row in df_group.iterrows():
        z_diff = row['Z-Score Difference (Senescence - Cell Cycle)']
        num_cells = row['Number of cells']
        cluster_val = row['Cluster']
        
        emp_cc = row['Cell Cycle Empirical P-value']
        emp_sen = row['Senescence Empirical P-value']
        comp_cc = row['Competitive P-value Cell Cycle']
        comp_sen = row['Competitive P-value Senescence']
        
        if pd.isna(z_diff):
            z_diff = 0
            
        z_diff_capped = 10 if z_diff >= 10 else (-10 if z_diff <= -10 else z_diff)
        z_diff_list.append(z_diff_capped)
        
        bubble_color = determine_activity_color(emp_cc, emp_sen, comp_cc, comp_sen)
        colors.append(bubble_color)
        
        if max_cells > 0:
            proportional_area = (num_cells / max_cells) * max_bubble_area
        else:
            proportional_area = 0
            
        scatter_sizes.append(proportional_area)
        
        # Annotation Logic
        if total_cells_in_dataset > 0:
            perc = (num_cells / total_cells_in_dataset) * 100
            perc_text = f"{perc:.1f}%"
            
            radius_pts = math.sqrt(proportional_area / math.pi)
            padding_pts = 6 
            
            if z_diff_capped < 0:
                x_offset = radius_pts + padding_pts
                ha = 'left'
            else:
                x_offset = -(radius_pts + padding_pts)
                ha = 'right'
                
            ax.annotate(
                perc_text,
                xy=(z_diff_capped, cluster_val),
                xytext=(x_offset, 0),
                textcoords='offset points', 
                ha=ha,
                va='center',
                fontsize=16,
                color='black',
                fontweight='medium'
            )
        
    ax.scatter(
        z_diff_list, 
        clusters_list, 
        s=scatter_sizes, 
        c=colors, 
        alpha=0.6, 
        edgecolors='black'
    )
    
    # Title and Labels
    ax.set_title(f"{clean_title} (k=8)", fontsize=24, fontweight='bold', ha='center', pad=15)   
    ax.set_xlabel('Effect Size Difference', fontsize=20)
    
    # Only show the Y-axis label on the very first (left-most) plot
    if i == 0:
        ax.set_ylabel('', fontsize=22)
    else:
        ax.set_ylabel('', fontsize=22)
    
    ax.set_xlim(-12, 12)
    ax.set_ylim(0.5, global_max_cluster + 0.5)
    
    ax.set_xticks(range(-10, 11, 5)) 
    ax.set_xticklabels([r'$\leq -10$' if x <= -10 else (r'$\geq 10$' if x >= 10 else str(x)) for x in range(-10, 11, 5)], fontsize=16)   
    
    ax.set_yticks(range(1, int(global_max_cluster) + 1))
    ax.set_yticklabels([f'Cluster {c}' for c in range(1, int(global_max_cluster) + 1)], fontsize=16)
    
    ax.axvline(x=0, color='grey', linewidth=0.8)
    ax.grid(True, linestyle='--', alpha=0.5, axis='x')
    
# Adjust spacing so the plots don't overlap
plt.tight_layout()

# Save the combined plot
save_path = os.path.join(data_dir, "Combined_Zscore_Diff_Plot_k8.png")
plt.savefig(save_path, dpi=400, bbox_inches='tight', facecolor='white')
print(f"Saved combined plot to {save_path}")

plt.close(fig)
print("\nCombined k=8 plot successfully generated and saved!")

# Final version

In [ ]:
# k = 4 with summary percentages 
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os

# Define the directory where your CSVs are saved
data_dir = '/path/to/xGATE/data/senescence_4/'

# Map the display titles to their corresponding file names
file_mapping = [
    {'title': 'CTRL',   'filename': 'competitive_analysis_summary_ctrl.csv'},
    {'title': 'ETO 0',  'filename': 'competitive_analysis_summary_eto_0.csv'},
    {'title': 'ETO 1',  'filename': 'competitive_analysis_summary_eto_1.csv'},
    {'title': 'ETO 2',  'filename': 'competitive_analysis_summary_eto_2.csv'},
    {'title': 'ETO 4',  'filename': 'competitive_analysis_summary_eto_4.csv'},
    {'title': 'ETO 7',  'filename': 'competitive_analysis_summary_eto_7.csv'},
    {'title': 'ETO 10', 'filename': 'competitive_analysis_summary_eto_10.csv'}
]

def determine_activity_color(emp_cc, emp_sen, comp_cc, comp_sen):
    if pd.isna(emp_cc) or pd.isna(emp_sen):
        return 'grey'
    if emp_cc >= 0.05 and emp_sen >= 0.05:
        return 'grey'
    elif emp_cc < 0.05 and emp_sen >= 0.05:
        return 'lightgreen'
    elif emp_sen < 0.05 and emp_cc >= 0.05:
        return 'skyblue'
    elif emp_cc < 0.05 and emp_sen < 0.05:
        if comp_cc < 0.05:
            return 'lightgreen'
        elif comp_sen < 0.05:
            return 'skyblue'
        else:
            return 'grey'
    return 'grey'

# =========================
# Standalone Legend Generation
# =========================
print("Generating standalone legend...")
legend_patches = [
    mpatches.Patch(color='skyblue', label='Cellular Senescence'),
    mpatches.Patch(color='lightgreen', label='Cell Cycle'),
    mpatches.Patch(color='grey', label='Not Significant')
]

fig_leg = plt.figure(figsize=(10, 2), facecolor='white') 
ax_leg = fig_leg.add_subplot(111)
ax_leg.axis('off') 
ax_leg.legend(handles=legend_patches, loc='center', ncol=3, frameon=False, fontsize=18)

legend_save_path = os.path.join(data_dir, "Standalone_Legend.png")
fig_leg.savefig(legend_save_path, dpi=300, bbox_inches='tight', facecolor='white')
plt.close(fig_leg)
print(f"Legend saved to {legend_save_path}")

# =========================
# Combined Plot Generation
# =========================
print("\nLoading data and generating combined plot...")

# First, filter out any files that don't exist and find the global max cluster for the shared Y-axis
valid_groups = []
global_max_cluster = 0

for group in file_mapping:
    filepath = os.path.join(data_dir, group['filename'])
    if os.path.exists(filepath):
        valid_groups.append(group)
        df = pd.read_csv(filepath)
        curr_max = df['Cluster'].max()
        if curr_max > global_max_cluster:
            global_max_cluster = curr_max
    else:
        print(f"Warning: File {filepath} not found. Skipping {group['title']}.")

# Fallback if empty
if pd.isna(global_max_cluster) or global_max_cluster == 0:
    global_max_cluster = 4

num_plots = len(valid_groups)

# Create a figure with 1 row and 'num_plots' columns. 
# Width is roughly 5 inches per plot, height is 6 inches.
fig, axes = plt.subplots(nrows=1, ncols=num_plots, figsize=(5 * num_plots, 6), sharey=True, facecolor='white')

# Ensure axes is iterable even if there's only 1 valid file
if num_plots == 1:
    axes = [axes]

for i, group in enumerate(valid_groups):
    ax = axes[i]
    filepath = os.path.join(data_dir, group['filename'])
    df = pd.read_csv(filepath)
    
    total_cells_in_dataset = df['Number of cells'].sum()
    max_cells = df['Number of cells'].max()
    max_bubble_area = 2500 
    
    clusters_list = df['Cluster'].tolist()
    z_diff_list = []
    colors = []
    scatter_sizes = []
    
    # Tracking counts for the summary block
    cc_count = 0
    sen_count = 0
    trans_count = 0
    
    for index, row in df.iterrows():
        z_diff = row['Z-Score Difference (Senescence - Cell Cycle)']
        num_cells = row['Number of cells']
        cluster_val = row['Cluster']
        
        emp_cc = row['Cell Cycle Empirical P-value']
        emp_sen = row['Senescence Empirical P-value']
        comp_cc = row['Competitive P-value Cell Cycle']
        comp_sen = row['Competitive P-value Senescence']
        
        if pd.isna(z_diff):
            z_diff = 0
            
        z_diff_capped = 10 if z_diff >= 10 else (-10 if z_diff <= -10 else z_diff)
        z_diff_list.append(z_diff_capped)
        
        bubble_color = determine_activity_color(emp_cc, emp_sen, comp_cc, comp_sen)
        colors.append(bubble_color)
        
        # Aggregate totals based on color
        if bubble_color == 'lightgreen':
            cc_count += num_cells
        elif bubble_color == 'skyblue':
            sen_count += num_cells
        elif bubble_color == 'grey':
            trans_count += num_cells
        
        if max_cells > 0:
            proportional_area = (num_cells / max_cells) * max_bubble_area
        else:
            proportional_area = 0
            
        scatter_sizes.append(proportional_area)
        
    ax.scatter(
        z_diff_list, 
        clusters_list, 
        s=scatter_sizes, 
        c=colors, 
        alpha=0.6, 
        edgecolors='black'
    )
    
    # --- ADD SUMMARY TEXT ---
    if total_cells_in_dataset > 0:
        cc_perc = (cc_count / total_cells_in_dataset) * 100
        trans_perc = (trans_count / total_cells_in_dataset) * 100
        sen_perc = (sen_count / total_cells_in_dataset) * 100
        
        summary_text = f"Cell Cycle: {cc_perc:.1f}%\nTransition: {trans_perc:.1f}%\nSenescence: {sen_perc:.1f}%"
        
        # Placed at y=-0.5 (deep into the new intentional padding area)
        ax.text(
            x=0, y=-0.5, 
            s=summary_text,
            ha='center', va='center',
            fontsize=18, color='black',
            bbox=dict(facecolor='white', alpha=1.0, edgecolor='none', pad=4) # alpha=1.0 totally hides the grid line behind the text
        )
    
    # Title and Labels
    ax.set_title(f"{group['title']} (k=4)", fontsize=24, fontweight='bold', ha='center', pad=15)   
    ax.set_xlabel('Effect Size Difference', fontsize=20)
    
    # Only show the Y-axis label on the very first (left-most) plot
    if i == 0:
        ax.set_ylabel('', fontsize=22)
    else:
        ax.set_ylabel('', fontsize=22)
    
    ax.set_xlim(-12, 12)
    
    # --- MASSIVE INTENTIONAL PADDING ADDED HERE ---
    # Dropped the bottom limit to -1.5. This guarantees over 2 full "cluster rows" of blank space
    ax.set_ylim(-1.5, global_max_cluster + 0.5)
    
    ax.set_xticks(range(-10, 11, 5)) 
    ax.set_xticklabels([r'$\leq -10$' if x <= -10 else (r'$\geq 10$' if x >= 10 else str(x)) for x in range(-10, 11, 5)], fontsize=16)   
    
    # Keep the actual y-ticks starting cleanly at 1
    ax.set_yticks(range(1, int(global_max_cluster) + 1))
    ax.set_yticklabels([f'Cluster {c}' for c in range(1, int(global_max_cluster) + 1)], fontsize=22, fontweight = 'bold')
    
    ax.axvline(x=0, color='grey', linewidth=0.8)
    ax.grid(True, linestyle='--', alpha=0.5, axis='x')
    
# Adjust spacing so the plots don't overlap
plt.tight_layout()

# Save the combined plot
save_path = os.path.join(data_dir, "Combined_Zscore_Diff_Plot_k4.png")
plt.savefig(save_path, dpi=400, bbox_inches='tight', facecolor='white')
print(f"Saved combined plot to {save_path}")

plt.close(fig)
print("\nCombined plot and standalone legend successfully generated and saved!")

In [ ]:
# k = 4 with summary percentages 
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os

# Define the directory where your CSVs are saved
data_dir = '/path/to/xGATE/data/senescence_6/'

# Map the display titles to their corresponding file names
file_mapping = [
    {'title': 'CTRL',   'filename': 'competitive_analysis_summary_ctrl_2.csv'},
    {'title': 'ETO 0',  'filename': 'competitive_analysis_summary_eto_0.csv'},
    {'title': 'ETO 1',  'filename': 'competitive_analysis_summary_eto_1.csv'},
    {'title': 'ETO 2',  'filename': 'competitive_analysis_summary_eto_2.csv'},
    {'title': 'ETO 4',  'filename': 'competitive_analysis_summary_eto_4.csv'},
    {'title': 'ETO 7',  'filename': 'competitive_analysis_summary_eto_7.csv'},
    {'title': 'ETO 10', 'filename': 'competitive_analysis_summary_eto_10.csv'}
]

def determine_activity_color(emp_cc, emp_sen, comp_cc, comp_sen):
    if pd.isna(emp_cc) or pd.isna(emp_sen):
        return 'grey'
    if emp_cc >= 0.05 and emp_sen >= 0.05:
        return 'grey'
    elif emp_cc < 0.05 and emp_sen >= 0.05:
        return 'lightgreen'
    elif emp_sen < 0.05 and emp_cc >= 0.05:
        return 'skyblue'
    elif emp_cc < 0.05 and emp_sen < 0.05:
        if comp_cc < 0.05:
            return 'lightgreen'
        elif comp_sen < 0.05:
            return 'skyblue'
        else:
            return 'grey'
    return 'grey'

# =========================
# Standalone Legend Generation
# =========================
print("Generating standalone legend...")
legend_patches = [
    mpatches.Patch(color='skyblue', label='Cellular Senescence'),
    mpatches.Patch(color='lightgreen', label='Cell Cycle'),
    mpatches.Patch(color='grey', label='Not Significant')
]

fig_leg = plt.figure(figsize=(10, 2), facecolor='white') 
ax_leg = fig_leg.add_subplot(111)
ax_leg.axis('off') 
ax_leg.legend(handles=legend_patches, loc='center', ncol=3, frameon=False, fontsize=18)

legend_save_path = os.path.join(data_dir, "Standalone_Legend.png")
fig_leg.savefig(legend_save_path, dpi=300, bbox_inches='tight', facecolor='white')
plt.close(fig_leg)
print(f"Legend saved to {legend_save_path}")

# =========================
# Combined Plot Generation
# =========================
print("\nLoading data and generating combined plot...")

# First, filter out any files that don't exist and find the global max cluster for the shared Y-axis
valid_groups = []
global_max_cluster = 0

for group in file_mapping:
    filepath = os.path.join(data_dir, group['filename'])
    if os.path.exists(filepath):
        valid_groups.append(group)
        df = pd.read_csv(filepath)
        curr_max = df['Cluster'].max()
        if curr_max > global_max_cluster:
            global_max_cluster = curr_max
    else:
        print(f"Warning: File {filepath} not found. Skipping {group['title']}.")

# Fallback if empty
if pd.isna(global_max_cluster) or global_max_cluster == 0:
    global_max_cluster = 4

num_plots = len(valid_groups)

# Create a figure with 1 row and 'num_plots' columns. 
# Width is roughly 5 inches per plot, height is 6 inches.
fig, axes = plt.subplots(nrows=1, ncols=num_plots, figsize=(5 * num_plots, 6), sharey=True, facecolor='white')

# Ensure axes is iterable even if there's only 1 valid file
if num_plots == 1:
    axes = [axes]

for i, group in enumerate(valid_groups):
    ax = axes[i]
    filepath = os.path.join(data_dir, group['filename'])
    df = pd.read_csv(filepath)
    
    total_cells_in_dataset = df['Number of cells'].sum()
    max_cells = df['Number of cells'].max()
    max_bubble_area = 2500 
    
    clusters_list = df['Cluster'].tolist()
    z_diff_list = []
    colors = []
    scatter_sizes = []
    
    # Tracking counts for the summary block
    cc_count = 0
    sen_count = 0
    trans_count = 0
    
    for index, row in df.iterrows():
        z_diff = row['Z-Score Difference (Senescence - Cell Cycle)']
        num_cells = row['Number of cells']
        cluster_val = row['Cluster']
        
        emp_cc = row['Cell Cycle Empirical P-value']
        emp_sen = row['Senescence Empirical P-value']
        comp_cc = row['Competitive P-value Cell Cycle']
        comp_sen = row['Competitive P-value Senescence']
        
        if pd.isna(z_diff):
            z_diff = 0
            
        z_diff_capped = 10 if z_diff >= 10 else (-10 if z_diff <= -10 else z_diff)
        z_diff_list.append(z_diff_capped)
        
        bubble_color = determine_activity_color(emp_cc, emp_sen, comp_cc, comp_sen)
        colors.append(bubble_color)
        
        # Aggregate totals based on color
        if bubble_color == 'lightgreen':
            cc_count += num_cells
        elif bubble_color == 'skyblue':
            sen_count += num_cells
        elif bubble_color == 'grey':
            trans_count += num_cells
        
        if max_cells > 0:
            proportional_area = (num_cells / max_cells) * max_bubble_area
        else:
            proportional_area = 0
            
        scatter_sizes.append(proportional_area)
        
    ax.scatter(
        z_diff_list, 
        clusters_list, 
        s=scatter_sizes, 
        c=colors, 
        alpha=0.6, 
        edgecolors='black'
    )
    
    # --- ADD SUMMARY TEXT ---
    if total_cells_in_dataset > 0:
        cc_perc = (cc_count / total_cells_in_dataset) * 100
        trans_perc = (trans_count / total_cells_in_dataset) * 100
        sen_perc = (sen_count / total_cells_in_dataset) * 100
        
        summary_text = f"Cell Cycle: {cc_perc:.1f}%\nTransition: {trans_perc:.1f}%\nSenescence: {sen_perc:.1f}%"
        
        # Placed at y=-0.5 (deep into the new intentional padding area)
        ax.text(
            x=0, y=-0.5, 
            s=summary_text,
            ha='center', va='center',
            fontsize=18, color='black',
            bbox=dict(facecolor='white', alpha=1.0, edgecolor='none', pad=4) # alpha=1.0 totally hides the grid line behind the text
        )
    
    # Title and Labels
    ax.set_title(f"{group['title']} (k=6)", fontsize=24, fontweight='bold', ha='center', pad=15)   
    ax.set_xlabel('Effect Size Difference', fontsize=20)
    
    # Only show the Y-axis label on the very first (left-most) plot
    if i == 0:
        ax.set_ylabel('', fontsize=22)
    else:
        ax.set_ylabel('', fontsize=22)
    
    ax.set_xlim(-12, 12)
    
    # --- MASSIVE INTENTIONAL PADDING ADDED HERE ---
    # Dropped the bottom limit to -1.5. This guarantees over 2 full "cluster rows" of blank space
    ax.set_ylim(-1.5, global_max_cluster + 0.5)
    
    ax.set_xticks(range(-10, 11, 5)) 
    ax.set_xticklabels([r'$\leq -10$' if x <= -10 else (r'$\geq 10$' if x >= 10 else str(x)) for x in range(-10, 11, 5)], fontsize=16)   
    
    # Keep the actual y-ticks starting cleanly at 1
    ax.set_yticks(range(1, int(global_max_cluster) + 1))
    ax.set_yticklabels([f'Cluster {c}' for c in range(1, int(global_max_cluster) + 1)], fontsize=22, fontweight = 'bold')
    
    ax.axvline(x=0, color='grey', linewidth=0.8)
    ax.grid(True, linestyle='--', alpha=0.5, axis='x')
    
# Adjust spacing so the plots don't overlap
plt.tight_layout()

# Save the combined plot
save_path = os.path.join(data_dir, "Combined_Zscore_Diff_Plot_k6.png")
plt.savefig(save_path, dpi=400, bbox_inches='tight', facecolor='white')
print(f"Saved combined plot to {save_path}")

plt.close(fig)
print("\nCombined plot and standalone legend successfully generated and saved!")

In [ ]:
# k = 8 with summary percentages in a Combined Subplot
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os
import math

# Define the paths
data_dir = '/path/to/xGATE/data/senescence_8/'
csv_file_path = os.path.join(data_dir, 'ETO_CTRL_Competitive_full_dataset_8_clusters.csv')

# Map the CSV's raw 'Group' names to the clean titles
# Using a dictionary here also safely defines the chronological order for the subplots
group_name_mapping = {
    'Ctrl': 'CTRL',
    'ETO Day 0': 'ETO 0',
    'ETO Day 1': 'ETO 1',
    'ETO Day 2': 'ETO 2',
    'ETO Day 4': 'ETO 4',
    'ETO Day 7': 'ETO 7',
    'ETO Day 10': 'ETO 10'
}

def determine_activity_color(emp_cc, emp_sen, comp_cc, comp_sen):
    if pd.isna(emp_cc) or pd.isna(emp_sen):
        return 'grey'
    if emp_cc >= 0.05 and emp_sen >= 0.05:
        return 'grey'
    elif emp_cc < 0.05 and emp_sen >= 0.05:
        return 'lightgreen'
    elif emp_sen < 0.05 and emp_cc >= 0.05:
        return 'skyblue'
    elif emp_cc < 0.05 and emp_sen < 0.05:
        if comp_cc < 0.05:
            return 'lightgreen'
        elif comp_sen < 0.05:
            return 'skyblue'
        else:
            return 'grey'
    return 'grey'

# =========================
# Standalone Legend Generation
# =========================
print("Generating standalone legend...")
legend_patches = [
    mpatches.Patch(color='skyblue', label='Cellular Senescence'),
    mpatches.Patch(color='lightgreen', label='Cell Cycle'),
    mpatches.Patch(color='grey', label='Not Significant')
]

fig_leg = plt.figure(figsize=(10, 2), facecolor='white')
ax_leg = fig_leg.add_subplot(111)
ax_leg.axis('off') 
ax_leg.legend(handles=legend_patches, loc='center', ncol=3, frameon=False, fontsize=24)

legend_save_path = os.path.join(data_dir, "Standalone_Legend_8_clusters.png")
fig_leg.savefig(legend_save_path, dpi=300, bbox_inches='tight', facecolor='white')
plt.close(fig_leg)
print(f"Legend saved to {legend_save_path}")

# =========================
# Combined Plot Generation
# =========================
print(f"\nLoading data from {csv_file_path}...")
df_full = pd.read_csv(csv_file_path)

# Filter groups to only those present in the CSV, keeping the explicit order of our mapping
valid_raw_groups = [g for g in group_name_mapping.keys() if g in df_full['Group'].unique()]

# Find the global max cluster for the shared Y-axis (fallback to 8)
global_max_cluster = df_full['Cluster'].max()
if pd.isna(global_max_cluster) or global_max_cluster == 0:
    global_max_cluster = 8

num_plots = len(valid_raw_groups)

# Create a figure with 1 row and 'num_plots' columns
fig, axes = plt.subplots(nrows=1, ncols=num_plots, figsize=(5 * num_plots, 6), sharey=True, facecolor='white')

# Ensure axes is iterable
if num_plots == 1:
    axes = [axes]

for i, raw_group_name in enumerate(valid_raw_groups):
    ax = axes[i]
    clean_title = group_name_mapping[raw_group_name]
    
    # Isolate data for just this group
    df_group = df_full[df_full['Group'] == raw_group_name].copy()
    
    total_cells_in_dataset = df_group['Number of cells'].sum()
    max_cells = df_group['Number of cells'].max()
    max_bubble_area = 2500 
    
    clusters_list = df_group['Cluster'].tolist()
    z_diff_list = []
    colors = []
    scatter_sizes = []
    
    # Tracking counts for the summary block
    cc_count = 0
    sen_count = 0
    trans_count = 0
    
    for index, row in df_group.iterrows():
        z_diff = row['Z-Score Difference (Senescence - Cell Cycle)']
        num_cells = row['Number of cells']
        cluster_val = row['Cluster']
        
        emp_cc = row['Cell Cycle Empirical P-value']
        emp_sen = row['Senescence Empirical P-value']
        comp_cc = row['Competitive P-value Cell Cycle']
        comp_sen = row['Competitive P-value Senescence']
        
        if pd.isna(z_diff):
            z_diff = 0
            
        z_diff_capped = 10 if z_diff >= 10 else (-10 if z_diff <= -10 else z_diff)
        z_diff_list.append(z_diff_capped)
        
        bubble_color = determine_activity_color(emp_cc, emp_sen, comp_cc, comp_sen)
        colors.append(bubble_color)
        
        # Aggregate totals based on color
        if bubble_color == 'lightgreen':
            cc_count += num_cells
        elif bubble_color == 'skyblue':
            sen_count += num_cells
        elif bubble_color == 'grey':
            trans_count += num_cells
        
        if max_cells > 0:
            proportional_area = (num_cells / max_cells) * max_bubble_area
        else:
            proportional_area = 0
            
        scatter_sizes.append(proportional_area)
        
    ax.scatter(
        z_diff_list, 
        clusters_list, 
        s=scatter_sizes, 
        c=colors, 
        alpha=0.6, 
        edgecolors='black'
    )
    
    # --- ADD SUMMARY TEXT ---
    if total_cells_in_dataset > 0:
        cc_perc = (cc_count / total_cells_in_dataset) * 100
        trans_perc = (trans_count / total_cells_in_dataset) * 100
        sen_perc = (sen_count / total_cells_in_dataset) * 100
        
        summary_text = f"Cell Cycle: {cc_perc:.1f}%\nTransition: {trans_perc:.1f}%\nSenescence: {sen_perc:.1f}%"
        
        # Placed at y=-0.5 (deep into the new intentional padding area)
        ax.text(
            x=0, y=-0.5, 
            s=summary_text,
            ha='center', va='center',
            fontsize=18, color='black',
            bbox=dict(facecolor='white', alpha=1.0, edgecolor='none', pad=4) # alpha=1.0 completely hides the grid line
        )
    
    # Title and Labels
    ax.set_title(f"{clean_title} (k=8)", fontsize=24, fontweight='bold', ha='center', pad=15)   
    ax.set_xlabel('Effect Size Difference', fontsize=20)
    
    # Only show the Y-axis label on the very first (left-most) plot
    if i == 0:
        ax.set_ylabel('', fontsize=22) # Fixed this so "Cluster" displays on the first plot
    else:
        ax.set_ylabel('', fontsize=22)
    
    ax.set_xlim(-12, 12)
    
    # --- INTENTIONAL PADDING FOR TEXT ---
    # Drop the bottom limit to -1.5 to make room for the text block safely away from bubbles
    ax.set_ylim(-1.5, global_max_cluster + 0.5)
    
    ax.set_xticks(range(-10, 11, 5)) 
    ax.set_xticklabels([r'$\leq -10$' if x <= -10 else (r'$\geq 10$' if x >= 10 else str(x)) for x in range(-10, 11, 5)], fontsize=16)   
    
    ax.set_yticks(range(1, int(global_max_cluster) + 1))
    ax.set_yticklabels([f'Cluster {c}' for c in range(1, int(global_max_cluster) + 1)], fontsize=22, fontweight = 'bold')
    
    ax.axvline(x=0, color='grey', linewidth=0.8)
    ax.grid(True, linestyle='--', alpha=0.5, axis='x')
    
# Adjust spacing so the plots don't overlap
plt.tight_layout()

# Save the combined plot
save_path = os.path.join(data_dir, "Combined_Zscore_Diff_Plot_k8.png")
plt.savefig(save_path, dpi=400, bbox_inches='tight', facecolor='white')
print(f"Saved combined plot to {save_path}")

plt.close(fig)
print("\nCombined k=8 plot successfully generated and saved!")

In [ ]:
from PIL import Image
import os

# Define file paths
paths = [
    "/path/to/xGATE/data/senescence_8/Standalone_Legend_8_clusters.png",
    "/path/to/xGATE/data/senescence_4/Combined_Zscore_Diff_Plot_k4.png",
    "/path/to/xGATE/data/senescence_6/Combined_Zscore_Diff_Plot_k6.png",
    "/path/to/xGATE/data/senescence_8/Combined_Zscore_Diff_Plot_k8.png"
]
output_path = "/path/to/xGATE/data/senescence/Stacked_Plots.pdf"

# Load images
images = [Image.open(p).convert("RGB") for p in paths]

# Calculate dimensions for the final canvas
# Width will be the width of the widest image
# Height will be the sum of all image heights
max_width = max(img.width for img in images)
total_height = sum(img.height for img in images)

# Create a white background canvas
canvas = Image.new("RGB", (max_width, total_height), (255, 255, 255))

# Paste images onto canvas with center alignment
current_y = 0
for img in images:
    # Calculate horizontal offset to center the image
    x_offset = (max_width - img.width) // 2
    canvas.paste(img, (x_offset, current_y))
    current_y += img.height

# Ensure output directory exists and save as PDF
os.makedirs(os.path.dirname(output_path), exist_ok=True)
canvas.save(output_path, "PDF", resolution=100.0)

print(f"File successfully saved to: {output_path}")

In [ ]:
import pandas as pd
import numpy as np

# Load the problematic dataset and the working dataset for comparison
df_sen = pd.read_csv('/path/to/xGATE/data/senescence_6/df_normalized_eto_7_cluster1_Cellular_senescence.csv')
df_cc = pd.read_csv('/path/to/xGATE/data/senescence_6/df_normalized_eto_7_cluster1_Cell_cycle.csv')

def diagnose_dataframe(df, name):
    print(f"--- Diagnostics for {name} ---")
    
    # Check for NaNs
    total_nans = df.isna().sum().sum()
    print(f"Total NaN values: {total_nans}")
    
    # Check for Infinity
    total_infs = np.isinf(df).sum().sum()
    print(f"Total Infinity values: {total_infs}")
    
    # Check the overall scale (Min/Max/Mean)
    print(f"Global Min: {df.min().min():.4f}")
    print(f"Global Max: {df.max().max():.4f}")
    print(f"Global Mean: {df.mean().mean():.4f}")
    
    # Check if any columns have exactly zero variance (constants)
    zero_var_cols = (df.var() == 0).sum()
    print(f"Columns with zero variance: {zero_var_cols} out of {df.shape[1]}")
    print("\n")

diagnose_dataframe(df_cc, "Cell Cycle (Working)")
diagnose_dataframe(df_sen, "Cellular Senescence (Failing)")

In [ ]:
import pandas as pd
import numpy as np

# Load the problematic dataset and the working dataset for comparison
df_sen = pd.read_csv('/path/to/xGATE/data/senescence_6/df_normalized_eto_4_cluster1_Cellular_senescence.csv')
df_cc = pd.read_csv('/path/to/xGATE/data/senescence_6/df_normalized_eto_10_cluster5_Cell_cycle.csv')

def diagnose_dataframe(df, name):
    print(f"--- Diagnostics for {name} ---")
    
    # Check for NaNs
    total_nans = df.isna().sum().sum()
    print(f"Total NaN values: {total_nans}")
    
    # Check for Infinity
    total_infs = np.isinf(df).sum().sum()
    print(f"Total Infinity values: {total_infs}")
    
    # Check the overall scale (Min/Max/Mean)
    print(f"Global Min: {df.min().min():.4f}")
    print(f"Global Max: {df.max().max():.4f}")
    print(f"Global Mean: {df.mean().mean():.4f}")
    
    # Check if any columns have exactly zero variance (constants)
    zero_var_cols = (df.var() == 0).sum()
    print(f"Columns with zero variance: {zero_var_cols} out of {df.shape[1]}")
    print("\n")

diagnose_dataframe(df_cc, "Cell Cycle (Working)")
diagnose_dataframe(df_sen, "Cellular Senescence (Failing)")

In [ ]:
import pandas as pd
import numpy as np

# Load the problematic dataset and the working dataset for comparison
df_sen = pd.read_csv('/path/to/xGATE/data/senescence_6/df_normalized_eto_1_cluster1_Cellular_senescence.csv')
df_cc = pd.read_csv('/path/to/xGATE/data/senescence_6/df_normalized_eto_10_cluster5_Cell_cycle.csv')

def diagnose_dataframe(df, name):
    print(f"--- Diagnostics for {name} ---")
    
    # Check for NaNs
    total_nans = df.isna().sum().sum()
    print(f"Total NaN values: {total_nans}")
    
    # Check for Infinity
    total_infs = np.isinf(df).sum().sum()
    print(f"Total Infinity values: {total_infs}")
    
    # Check the overall scale (Min/Max/Mean)
    print(f"Global Min: {df.min().min():.4f}")
    print(f"Global Max: {df.max().max():.4f}")
    print(f"Global Mean: {df.mean().mean():.4f}")
    
    # Check if any columns have exactly zero variance (constants)
    zero_var_cols = (df.var() == 0).sum()
    print(f"Columns with zero variance: {zero_var_cols} out of {df.shape[1]}")
    print("\n")

diagnose_dataframe(df_cc, "Cell Cycle (Working)")
diagnose_dataframe(df_sen, "Cellular Senescence (Failing)")